# Exploring the StatsBomb data yourself

This is the general recipe for poking at any new JSON dataset: load it, check its shape, look at one record closely, then flatten it into a table so you can use pandas to slice/filter/count.

Run each cell with `Shift+Enter`. Change things and re-run — that's the whole point of a notebook, nothing here is final.

In [1]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/data")
MATCH_ID = 8658  # 2018 World Cup Final: France 4-2 Croatia

## Step 1: Load it raw, check the shape

Before doing anything clever: what type of object is this (list? dict?), how big is it, what does one element look like?

In [2]:
with open(DATA_DIR / "events" / f"{MATCH_ID}.json", encoding="utf-8") as f:
    events = json.load(f)

print(type(events))
print("number of events:", len(events))

<class 'list'>
number of events: 2978


In [3]:
# Look at ONE event closely. This is how you learn the schema —
# not from documentation, from just reading a real example.
events[0]

{'id': '4a44199a-3111-4e28-b567-9c1393e68dff',
 'index': 1,
 'period': 1,
 'timestamp': '00:00:00.000',
 'minute': 0,
 'second': 0,
 'type': {'id': 35, 'name': 'Starting XI'},
 'possession': 1,
 'possession_team': {'id': 771, 'name': 'France'},
 'play_pattern': {'id': 1, 'name': 'Regular Play'},
 'team': {'id': 771, 'name': 'France'},
 'duration': 0.0,
 'tactics': {'formation': 442,
  'lineup': [{'player': {'id': 3099, 'name': 'Hugo Lloris'},
    'position': {'id': 1, 'name': 'Goalkeeper'},
    'jersey_number': 1},
   {'player': {'id': 5476, 'name': 'Benjamin Pavard'},
    'position': {'id': 2, 'name': 'Right Back'},
    'jersey_number': 2},
   {'player': {'id': 5485, 'name': 'Raphaël Varane'},
    'position': {'id': 3, 'name': 'Right Center Back'},
    'jersey_number': 4},
   {'player': {'id': 5492, 'name': 'Samuel Yves Umtiti'},
    'position': {'id': 5, 'name': 'Left Center Back'},
    'jersey_number': 5},
   {'player': {'id': 5484, 'name': 'Lucas Hernández Pi'},
    'position': {'i

## Step 2: Flatten into a table

Each event is a nested dict (dicts inside dicts). `pd.json_normalize` flattens that into columns like `type.name`, `player.name`, `shot.outcome.name`, etc, so you get one row per event.

In [4]:
df = pd.json_normalize(events)
print(df.shape)  # (rows, columns)
df.head()

(2978, 102)


,id,index,period,timestamp,minute,second,possession,duration,type.id,type.name,...,goalkeeper.technique.id,goalkeeper.technique.name,goalkeeper.body_part.id,goalkeeper.body_part.name,pass.deflected,dribble.overrun,substitution.outcome.id,substitution.outcome.name,substitution.replacement.id,substitution.replacement.name
0,4a44199a-3111-4e28-b567-9c1393e68dff,1,1,00:00:00.000,0,0,1,0.000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2137ed27-042c-4d73-877c-75d04406617c,2,1,00:00:00.000,0,0,1,1.412,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,531c8cbd-7a3e-4e46-91c1-549a98c27bcb,3,1,00:00:00.000,0,0,1,0.000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9b9060d4-13a5-4a44-aafa-2221ed10a8bc,4,1,00:00:00.000,0,0,1,0.000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6d7450f7-2590-4b04-a2ae-9ed11602e342,5,1,00:00:00.400,0,0,2,1.159,30,Pass,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# All the column names — scan this to see what's actually available per event.
list(df.columns)

['id',
 'index',
 'period',
 'timestamp',
 'minute',
 'second',
 'possession',
 'duration',
 'type.id',
 'type.name',
 'possession_team.id',
 'possession_team.name',
 'play_pattern.id',
 'play_pattern.name',
 'team.id',
 'team.name',
 'tactics.formation',
 'tactics.lineup',
 'related_events',
 'location',
 'player.id',
 'player.name',
 'position.id',
 'position.name',
 'pass.recipient.id',
 'pass.recipient.name',
 'pass.length',
 'pass.angle',
 'pass.height.id',
 'pass.height.name',
 'pass.end_location',
 'pass.type.id',
 'pass.type.name',
 'pass.body_part.id',
 'pass.body_part.name',
 'carry.end_location',
 'under_pressure',
 'pass.outcome.id',
 'pass.outcome.name',
 'ball_receipt.outcome.id',
 'ball_receipt.outcome.name',
 'duel.type.id',
 'duel.type.name',
 'pass.aerial_won',
 'duel.outcome.id',
 'duel.outcome.name',
 'counterpress',
 'interception.outcome.id',
 'interception.outcome.name',
 'pass.switch',
 'pass.cross',
 'dribble.outcome.id',
 'dribble.outcome.name',
 'foul_committ

## Step 3: Ask it questions

`value_counts()` is your best friend for a first look at any categorical column — it tells you what values exist and how common each is.

In [6]:
# What event types occur, and how often?
df["type.name"].value_counts()

type.name
Pass                 846
Ball Receipt*        747
Carry                617
Pressure             254
Ball Recovery         88
Duel                  53
Clearance             37
Camera On             34
Miscontrol            33
Block                 33
Goal Keeper           32
Foul Committed        28
Dribble               28
Dispossessed          27
Foul Won              27
Shot                  23
Dribbled Past         18
Interception          17
Injury Stoppage        6
Camera off             5
Substitution           5
Half Start             4
Half End               4
Starting XI            2
Shield                 2
Referee Ball-Drop      2
Own Goal For           1
Own Goal Against       1
Player Off             1
Player On              1
Error                  1
Tactical Shift         1
Name: count, dtype: int64

In [7]:
# Filter rows like any pandas table: just the shots.
shots = df[df["type.name"] == "Shot"]
shots[["minute", "player.name", "team.name", "shot.outcome.name", "shot.statsbomb_xg"]]

,minute,player.name,team.name,shot.outcome.name,shot.statsbomb_xg
723,20,Domagoj Vida,Croatia,Off T,0.127957
792,23,Ivan Rakitić,Croatia,Off T,0.069509
910,27,Ivan Perišić,Croatia,Goal,0.083949
1052,37,Antoine Griezmann,France,Goal,0.783500
1076,39,Ante Rebić,Croatia,Wayward,0.102178
1195,42,Ivan Perišić,Croatia,Blocked,0.112243
1200,42,Dejan Lovren,Croatia,Blocked,0.046405
1230,45,Domagoj Vida,Croatia,Off T,0.140840
1404,46,Antoine Griezmann,France,Saved,0.033398
1458,47,Ante Rebić,Croatia,Saved,0.072852


In [13]:
# Just the goals.
goals = shots[shots["shot.outcome.name"] == "Goal"]
goals[["minute", "player.name", "team.name"]]

# All shots including the missed ones
all_shots = shots[shots["shot.type.name"] == "Shot"]
all_shots[["minute", "player.name", "team.name"]]


,minute,player.name,team.name


## Try it yourself

Some things to try in a new cell below:
- `df["player.name"].value_counts()` — who's involved in the most events?
- `df[df["type.name"] == "Pass"]["pass.outcome.name"].value_counts()` — how do passes fail?
- Change `MATCH_ID` at the top to a different match and re-run everything (`Run All`) to compare.